# The normal brain across cohorts

What the cohorts' brain-state dynamics **share**, more than how they differ. One state set (the
same atlas × window × state definition for everyone), every cohort's rest and movie scans, all
subjects (no phenotype split: these are the cohorts as they are).

* **Train and projected.** The states are fitted on the training cohorts and every other cohort is
  labelled by that fit, never seen by it (`cluster.py`, `DEFAULT_TRAIN`). A projected cohort that
  looks like the training ones is evidence that the states generalise.
* **Equivalence, not "no difference".** A non-significant difference does not show that two cohorts
  agree. Figures 2 and 4 use a TOST (two one-sided tests, Lakens 2017): a cohort is **equivalent** to
  the others on a variable when the 90% CI of its difference lies inside ± SESOI, the smallest
  difference that would matter. SESOI = `sesoi_d` × the pooled SD of that variable; 0.2 is Cohen's
  "small effect", a convention to justify (or replace by a bound in real units). A small cohort has a
  wide CI and may not be shown equivalent: that is not evidence of a difference.

The tables come from `cohort_tables.py` and the figures from `cohort_plots.py`; the state diagram is
notebook 10's, with the same label rules. Nothing here writes to `outputs/`.

In [ ]:
import cohort_tables as ct, cohort_plots as cp

cfg = ct.Settings(
    space=dict(atlas="yeo7", window_s=30.0, states="HMM1_pca3_8"),   # see the list printed below
    # label -> the cohort= directories it reads, and its role in the state fit. A new cohort is one line.
    cohorts={"ds002837": dict(dirs=["ds002837"], role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan": dict(dirs=["camcan", "camcan_rest"], role="projected")},
    rest_pattern=r"rest",          # a task= that IS this (whole name, any case) is rest; every other is movie
    # figure 1: movie vs rest
    alpha=0.10,                    # BH-FDR
    fc_task=1.0,                   # |log2FC| for the graph (1 = twofold)
    fc_dyn=0.25,                   # |log2FC| for the dynamics
    edge_top=12,                   # the mean's strongest arrows, per task
    # figures 2 and 4: equivalence
    sesoi_d=0.2,                   # TOST bounds, in pooled SDs
    top_transitions=8,             # figure 2: the strongest transitions shown
    save_dir=None,                 # e.g. "figures/11" to save PNG + SVG
)
sets = ct.available_sets(cfg.root, cfg.cohorts)
print(f"{len(sets)} state sets have every cohort; e.g.")
for _, r in sets.head(8).iterrows():
    print(f"  space=dict(atlas={r.atlas!r}, window_s={r.window_s}, states={r.states!r})")
co = ct.Cohorts(cfg)
# What is read, from where, and as which kind: check this before reading any figure.
co.tasks()

## 1. Rest and movie, per cohort and together

Rows are the cohorts, then their mean (every cohort weighs once); columns are rest and movie. Each
diagram is the cohort's own dynamics, averaged over its subjects (a subject's runs or films first).
**Circle area** is time in a state, **the loop** is p(stay), **arrow width** is p(j | i), on one
width scale for the whole figure; every diagram draws the mean's `edge_top` strongest arrows of each
task. **Blue** on the movie is what differs from the **same people's** rest: paired t-test (Welch if
a cohort's rest and movie subjects do not overlap), BH-FDR over every variable, and |log2FC| ≥
`fc_task` (`fc_dyn` for the dynamics), with the change in pp. Beside each diagram are the dynamics:
⇄ switch rate, stopwatch = mean dwell, H entropy rate, ✳ dispersion, # distinct transitions.

In [ ]:
graphs = co.graphs()
marks, dyn = co.task_change()
cp.fig_diagrams(co, graphs, marks, dyn)
# marks[marks["hit"]].sort_values(["cohort", "p"])

## 2. Is every cohort equivalent to the others?

Movie only (the task every cohort shares). The transitions are the `top_transitions` strongest;
every occupancy and every dynamics variable is shown, the dynamics each on its own axis.

**2a, equivalence (TOST), in the variables' own units.** Each cohort's mean with its 90% CI (time in
% for the probabilities, seconds or bits for the dynamics). The grey band is the mean of the cohorts
± SESOI. **Filled** = equivalent to all the others pooled (the 90% CI of the difference lies inside
± SESOI), **hollow** = not shown equivalent; **✓** at the right when every cohort is.

**2b, effect size (Cohen's d).** The same comparison as a standardised difference with its 95% CI,
on bands at |d| 0.2 (small), 0.5 (medium), 0.8 (large). A large cohort is never "equivalent" by
accident, and a tiny one rarely is (its CI is wide); d says how far apart the cohorts are,
whatever their size.

In [ ]:
agree = co.agreement()
cp.fig_tost(co, agree)
# agree.groupby("variable")["equivalent"].all().value_counts()

In [ ]:
d = co.cohen()
cp.fig_cohen(co, d)
# d.pivot_table(index="variable", columns="cohort", values="d").round(2)

## 3. Does each projected cohort reproduce the training cohorts?

Movie, one row per projected cohort. Every variable's mean in the projected cohort (y) against its
mean in each training cohort (x, a dot per training cohort, in its colour): on the diagonal, the
projection reproduces the training. The title gives the correlation with each training cohort and
between the training cohorts: a projected cohort that matches them about as well as they match each
other is as close as two training cohorts are. The dynamics are log-log (they span 0.4 to ~35).

In [ ]:
means = co.cohort_means()
cp.fig_train_vs_projected(co, means)
# means.T

## 4. The transitions as matrices

One matrix per group (p(j | i), on one colour scale), with the time in each state on top and the
dynamics below, then a last panel testing every cell, occupancy and dynamics: each group vs all the
others pooled. Three switches:

* `GROUP_BY = "cohort"` compares the cohorts' movie; `"task"` compares tasks, with `TASKS = "kind"`
  (rest vs movie) or `"film"` (every movie task apart: films, episodes). `WITHIN = "camcan"` keeps a
  task comparison inside one cohort; `None` pools every cohort.
* `TEST = "tost"` (equivalence: ✓ where every group is equivalent to the others within ± `sesoi_d`
  SD, a red frame with k/N where only k are), `"cohen"` (the group furthest from the others, |d|
  coloured small / medium / large) or `None` (the matrices only).
* `SD = True` writes each cell's between-subject SD (pp), ± SD bars on the occupancy and mean ± SD
  for the dynamics.

Matrices that look alike can still fail the TOST: the bounds are ± `sesoi_d` × the between-subject
SD, and an HMM's p(j | i) varies little between subjects, so ± 0.2 SD can be a fraction of a pp.
Rest and movie are mostly the same people: the tests treat the groups as independent, which makes
equivalence harder to show, not easier.

In [ ]:
GROUP_BY = "cohort"            # "cohort" | "task"
TASKS = "kind"                 # with "task": "kind" (rest vs movie) | "film" (each movie task)
WITHIN = None                  # with "task": one cohort label, or None = every cohort pooled
TEST = "tost"                  # "tost" | "cohen" | None
SD = False
groups = co.groups(GROUP_BY, TASKS, WITHIN)
cp.fig_matrices(co, groups, TEST, SD, title=f"4  transitions by {GROUP_BY}", name=f"4_matrix_{GROUP_BY}")

In [ ]:
# the same figure, Cohen's d and with the SD
cp.fig_matrices(co, groups, "cohen", sd=True, title=f"4  transitions by {GROUP_BY}", name=f"4_matrix_{GROUP_BY}")

### 4, by task: rest vs movie

Every cohort's rest against every cohort's movie (`WITHIN` narrows it to one cohort). Change
`TASKS` to `"film"` to set the films (and episodes) against each other instead.

In [ ]:
task_groups = co.groups("task", tasks="kind", cohort=None)
cp.fig_matrices(co, task_groups, "cohen", title="4  rest vs movie", name="4_matrix_task_kind")

In [ ]:
film_groups = co.groups("task", tasks="film", cohort=None)
cp.fig_matrices(co, film_groups, "tost", title="4  the films against each other", name="4_matrix_task_film")

## 5. The latent space, per cohort

Every window of every cohort in the stage-4 latent space (`latents/`), one 3D panel per cohort on
the same axes. The embedding is fitted on the training cohorts and the others are projected onto
it, so a projected cohort filling the same space as the training ones is the same claim as figure 3,
seen window by window. At most `MAX_WINDOWS` windows per person are drawn (chosen at random), so a
cohort with long scans does not bury the others.

`BY` picks the colouring: `"participant"` (a gradient, one colour per person: are the people spread
over the whole space or each in a corner?), `"task"` (rest, movie and each film), `"cluster"` (the
state of `space`, needs the clustering stage) or `"single"` (one person per cohort, `WHO =
{cohort: subject id}`, in colour with their path, over everyone else in grey; a cohort left out shows
its first subject). `PATHS = True` joins each person's windows in time order. The figure is plotly:
drag to rotate, scroll to zoom, hover for the subject and task; with `save_dir` it is also written
as an .html. Needs `pip install --user plotly`.

In [ ]:
EMBEDDING = "umap3"            # or "pca3"
BY = "participant"             # "participant" | "task" | "cluster" | "single"
WHO = {}                       # with BY="single": {cohort: subject id}, e.g. {"camcan": "CC110033"}
PATHS = False
MAX_WINDOWS = 150
lat = ct.load_latents(co, EMBEDDING, MAX_WINDOWS)
cp.fig_latents(co, lat, by=BY, who=WHO, embedding=EMBEDDING, paths=PATHS).show()
# lat.groupby(["cohort", "kind"])["sub"].nunique()